
# 1. Plain Monte Carlo on the Ising model — and where it falls short

The journey starts *before* flatwalk. We sample the 2D Ising model the
textbook way — fixed-temperature Metropolis Monte Carlo — measure its energy,
and watch it run into the wall that flat-histogram sampling exists to knock
down: a canonical run only ever visits the energies that matter *at its own
temperature*. It can neither reach the rare states in the tails nor tell you
anything about a *different* temperature without starting over.

The maths of importance sampling and the Metropolis rule is in
:doc:`/theory/02-monte-carlo`; here we just feel the problem.


## The model

We reuse the energy and the single-spin-flip ``ΔE`` from
``examples/ising.py`` so the physics is identical to every later tutorial —
only the *sampling* changes from here on. ``conf.py`` puts the repo's
``examples/`` on ``sys.path`` for the docs build; the ``try`` block is only
for running this script standalone.



In [ ]:
import sys

try:
    from pathlib import Path

    sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "examples"))
except NameError:
    pass  # sphinx-gallery exec context: __file__ undefined, sys.path is already set

import ising  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

L = 6  # 36 spins; energies run over E ∈ [-2L², 2L²] = [-72, 72]

## A textbook Metropolis sweep

Propose a single-spin flip, accept it with probability ``min(1, e^{−βΔE})``.
We flip in place and track the running energy, recording ``E`` after every
step so we can see *which* energies the walk actually visits.



In [ ]:
def metropolis(beta, n_steps, rng):
    spins = ising.random_state(L, rng)[0]
    E = ising.total_energy(spins)
    energies = np.empty(n_steps)
    for t in range(n_steps):
        i = int(rng.integers(0, L))
        j = int(rng.integers(0, L))
        s = int(spins[i, j])
        nb = int(
            spins[(i - 1) % L, j]
            + spins[(i + 1) % L, j]
            + spins[i, (j - 1) % L]
            + spins[i, (j + 1) % L]
        )
        dE = 2.0 * s * nb
        if dE <= 0 or rng.random() < np.exp(-beta * dE):
            spins[i, j] = -s
            E += dE
        energies[t] = E
    return energies

## Run at three temperatures

Below, around, and above the ``L → ∞`` critical point ``T_c ≈ 2.27``. We
discard a burn-in and keep the rest as the equilibrium sample at each ``T``.



In [ ]:
rng = np.random.default_rng(0)
temperatures = [1.5, 2.27, 3.5]
n_steps = 60_000
burn_in = 10_000

samples = {}
for T in temperatures:
    e = metropolis(1.0 / T, n_steps, rng)[burn_in:]
    samples[T] = e
    print(f"T = {T:>4}:  ⟨E⟩ = {e.mean():7.1f}   (E ranges over [-72, 72])")

## Each temperature sees only its own sliver of the energy axis

The shaded band is the *full* range of energies the model can take. Each
canonical run piles up in a narrow window around its own equilibrium energy
and never leaves it. Nothing here knows the density of states ``g(E)``; each
run is a separate experiment, and the cold-tail and hot-tail energies are
simply never sampled.



In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.axvspan(-2 * L * L, 2 * L * L, color="0.9", label="full energy range")
for T, e in samples.items():
    ax.hist(e, bins=40, density=True, histtype="step", lw=1.5, label=f"T = {T}")
ax.set_xlabel("E")
ax.set_ylabel("visited fraction (per T)")
ax.set_title(f"L={L} Ising: plain Metropolis only samples near ⟨E⟩(T)")
ax.set_xlim(-2 * L * L - 5, 2 * L * L + 5)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## The wall

To draw the heat-capacity curve ``C_V(T)`` you would repeat this at dozens of
temperatures; to study a nucleation barrier you would need the rare states in
the tails that *no* canonical run visits; and reweighting one run to a
far-off temperature fails precisely because the energies that matter there
were never sampled. Each is the same shortcoming wearing a different hat: a
canonical sampler is tied to one temperature.

What if a *single* run instead estimated ``g(E)`` over the whole axis — and
with it every temperature at once? That is the promise of flat-histogram
sampling, and the next tutorial makes good on it with Wang-Landau.

